# Rat fine-tuning

Fine-tune a pretrained mouse model on normalized **D2**, using the shared experiment core. D2 currently has five rats and 20 longitudinal scans; scan counts vary by rat. Split **animals**, keeping every timepoint of a rat together. Validation selects the checkpoint; test rats are evaluated after fitting.

Run in the **rnet** environment. First inspect the data and split with training disabled. Supply a mouse checkpoint and review its architecture and output labels before enabling training. This notebook contains no separate training loop.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Dataset, checkpoint and fine-tuning settings

`MOUSE_CHECKPOINT` is deliberately unset. For `TRANSFER_MODE="weights"`, select a complete `best.weights.h5` or inference `model.keras` from a compatible mouse model. This retains **all pretrained heads**, with all layers trainable by default. Match its architecture, filter widths, component flags and label meanings to the settings below. A matching channel count alone does not establish matching label semantics; inspect the source run's `experiment.json`.

`TRANSFER_MODE="backbone"` instead requires a complete inference model and validates its feature graph before transfer. It **reinitializes all output heads**; use this explicitly when the target vocabulary changes. `FREEZE_BACKBONE=True` freezes feature layers including batch normalization, while keeping heads trainable. It does not implement an automatic freeze/unfreeze schedule.

The default task matches notebook 1: background, lesion, merged lateral ventricles and third ventricle. Normalized D2 annotation metadata covers these labels. The original 3DR-Net widths and zero-dropout behavior are retained. The learning rate below is an editable fine-tuning setting, not a tuned optimum. The reference and spacing use the mouse preprocessing convention; check the validation preview for the larger rat anatomy.

`postprocessing_settings` cleans only the binary brain mask and separates merged ventricles using the historical voxel-wise split. `original_label_ids=True` restores ROI IDs 0, 1, 3, 13, 21 for metrics, saved masks and the viewer. Morphology thresholds are voxel counts on the processed grid. GT retains the original sides. The historical fallback hemisphere means the side of array origin; a missing predicted midline raises an explicit error.


In [ ]:
from typing import Literal

from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
MOUSE_CHECKPOINT: Path | None = None
TRANSFER_MODE: Literal["weights", "backbone"] = "weights"
FREEZE_BACKBONE: bool = False
RUN_TRAINING: bool = False
RESUME_RUN: Path | None = None
SPLIT_FILE: Path | None = None

initialization = InitializationConfig()
if MOUSE_CHECKPOINT is not None:
    initialization = InitializationConfig(
        mode=TRANSFER_MODE, checkpoint=MOUSE_CHECKPOINT,
        freeze_backbone=FREEZE_BACKBONE,
    )
else:
    print("Data preview only: set MOUSE_CHECKPOINT before fine-tuning.")

training = TrainingConfig(
    model_name="d2_3drnet_finetuning", epochs=100, batch_size=4,
    seed=TRAINING_SEED, data_multiplier=1, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
# Fractions of all animals; saved split files keep their exact memberships.
split_settings = FixedSplitConfig(
    validation_fraction=0.2, test_fraction=0.2,
)

data_settings = DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D2", annotation_scope="full_volume"),))
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(112, 112, 112), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.5,
)
# Sample once; validation/test reuse evaluation_patches.json across epochs and runs.
evaluation_patch_settings = EvaluationPatchConfig(
    patches_per_scan=3, sampling="center_biased", seed=42,
    left_right_pairs=False,  # D1/rats: optional paired LR views; MouseFull: keep False.
)

model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    residual=True, attention=True, max_pooling=True, classifier=False,
)
labels = LabelConfig()
loss_settings = LossConfig(regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    evaluation_patches=evaluation_patch_settings,
    fixed_split=split_settings,
    model=model_settings, labels=labels, loss=loss_settings,
    inference=inference_settings, initialization=initialization,
    learning_rate=0.0001, split_seed=42,
)
config.validate()

## Configure callbacks directly

Fixed-split training uses the original LR reduction, TensorBoard histograms, early stopping and CSV logging. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

# Relative logging paths are resolved inside this run's fold directory.
log_dir: str = "tensorboard"
csv_filename: str = "training.csv"
csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)

callbacks: list[tf.keras.callbacks.Callback] = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=10,
        min_lr=1e-7, verbose=1,
    ),
    tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=20, verbose=1,
        restore_best_weights=False,  # The runner reloads the best saved weights before evaluation.
    ),
    csv_logger,
]


## Animal-level split and longitudinal coverage

Without a saved split, the deterministic default gives **three training rats, one validation rat and one test rat**. With such a small cohort, this is one exploratory fixed split, not a robust population estimate. Load a scientifically chosen `SPLIT_FILE` for manuscript experiments and reuse it for matched comparisons.

`RESUME_RUN` automatically loads that run's saved split. Keep the original checkpoint, configuration and source data unchanged when resuming. For grouped cross-validation use notebook 2's shared runner with D2 settings; keep independent validation animals inside each fold.

Counts below show animals and scans separately and include every discovered timepoint, including 45 days when present. `data_multiplier` repeats only training scans. It does not create new animals or duplicate validation/test scans.

In [ ]:
from IPython.display import display
from src.experiments.data import SubjectSplit, prepare_data

prepared = prepare_data(config)

if RESUME_RUN is not None:
    saved_split = RESUME_RUN / "splits.json"
    split = SubjectSplit.load(saved_split, prepared.subjects)
    if SPLIT_FILE is not None:
        requested_split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)
        if requested_split != split:
            raise ValueError("SPLIT_FILE differs from the resumed run's split")
elif SPLIT_FILE is not None:
    split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)
else:
    split = prepared.make_split()

display(prepared.split_summary(split))
manifest = prepared.subjects.manifest(prepared.subjects.subject_ids)
columns = ["Subject", "OriginalID", "Timepoint", "Scan"]
display(manifest[columns])

## Preview deterministic validation preprocessing

Inputs reorient to the shared reference first. Intensities use linear interpolation; masks use nearest-neighbour interpolation. Validation reuses saved centre-biased patches, while training uses augmented centre-biased random crops and full-volume inference covers the complete scan. Inspect a central slice below; the cropped preview does not establish whole-brain coverage.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "vscode"

validation_generator = prepared.make_generator(split.validation, training=False, batch_size=1)
images, targets = validation_generator[0]
print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Validation batches:", len(validation_generator))

slice_index: int = images.shape[3] // 2
image_slice = images[0, :, :, slice_index, 0].T
roi_labels = np.argmax(targets[0][0], axis=-1)
roi_slice = roi_labels[:, :, slice_index].T

image_trace = go.Heatmap(z=image_slice, colorscale="Gray", showscale=False)
figure = go.Figure(data=[image_trace])
figure.update_layout(title="Validation input: central slice", yaxis_scaleanchor="x")
figure.show()

maximum_label: int = len(config.labels.names) - 1
label_trace = go.Heatmap(z=roi_slice, zmin=0, zmax=maximum_label, colorscale="Viridis")
label_figure = go.Figure(data=[label_trace])
label_figure.update_layout(title="Mapped validation ROI labels", yaxis_scaleanchor="x")
label_figure.show()

del images, targets, validation_generator

## Fine-tune or resume

Enabling training requires pretrained initialization; the preview configuration cannot accidentally launch scratch training. The shared runner loads and audits the transfer, compiles with a fresh optimizer and saves the run provenance. Resume restores the fine-tuning optimizer and callback state at the last committed epoch rather than restarting optimization.

After validation selects the best checkpoint, test scans are predicted once per evaluation pass. Metrics use physical spacing, and saved masks return to the original rat scan's grid with **original annotation IDs**. Test results are not used to choose the checkpoint.

In [ ]:
from src.experiments.training import run_training

result = None
if RUN_TRAINING:
    if config.initialization.mode == "scratch":
        raise ValueError("Set MOUSE_CHECKPOINT and rerun configuration before fine-tuning")

    result = run_training(prepared, split, resume_run=RESUME_RUN, callbacks=callbacks)
    print("Fine-tuning run saved to:", result.run_dir)
else:
    print("Checks complete. Set a mouse checkpoint and RUN_TRAINING = True to fine-tune.")

## Inspect results by rat and timepoint

Training curves and per-scan test metrics are shown below. Keep timepoints attached to their rat when interpreting or exporting results: multiple scans from one test rat are **one biological test animal**, not independent samples. Animal-aware confidence intervals and comparisons belong in the results notebook.

The shared output directory includes `experiment.json`, `subjects.csv`, `splits.json`, `fold_0/initialization.json`, fitting recovery state, best weights, the inference model and full-volume metric/prediction exports. The initialization audit records the source checkpoint hash and copied/frozen layers.

In [ ]:
import pandas as pd
from src.experiments.training import history_figure

if result is not None:
    history = history_figure(result)
    history.show()
    display(pd.DataFrame(result.timings))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        test_metrics = metrics.loc[metrics["DatasetRole"] == "test"]
        dice = test_metrics.loc[test_metrics["Metric"] == "Dice"]
        columns = ["Subject", "Scan", "Region", "Value", "Status"]
        display(dice[columns])
        print("Full-volume metrics:", result.run_dir / "regions")

## Explore test predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_ROLE: str = 'test'

viewer_role = VIEW_ROLE
viewer_run = VIEW_RUN
current_result = globals().get("result")
if viewer_run is None and current_result is not None:
    viewer_run = current_result.run_dir
if viewer_run is None and RESUME_RUN is not None:
    viewer_run = RESUME_RUN

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This performs full-volume inference only, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = 'test'

checkpoint_run = CHECKPOINT_RUN
current_result = globals().get("result")
if checkpoint_run is None and current_result is not None:
    checkpoint_run = current_result.run_dir
if checkpoint_run is None:
    checkpoint_run = globals().get("RESUME_RUN")

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")
